In [2]:
import pandas as pd

In [3]:
CRISPRGeneEffect = pd.read_csv("CRISPRGeneEffect.csv")
Model = pd.read_csv("Model.csv")
OmicsExpressionTPMLogp1HumanProteinCodingGenes = pd.read_csv("OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv")
OmicsSomaticMutations = pd.read_csv("OmicsSomaticMutations.csv")

/var/folders/fm/__xjg2_1199f_fdhggm2cgd40000gn/T/ipykernel_45185/2310831729.py:4: DtypeWarning: Columns (0: DbsnpRsID, 1: TranscriptLikelyLof, 2: CivicDescription, 3: HessDriver, 4: HessSignature, 5: PharmgkbId, 6: GwasDisease, 7: GtexGene) have mixed types. Specify dtype option on import or set low_memory=False.
  OmicsSomaticMutations = pd.read_csv("OmicsSomaticMutations.csv")


In [4]:
display("CRISPR Gene Effect", CRISPRGeneEffect.head())

'CRISPR Gene Effect'

,Unnamed: 0,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550),A4GALT (53947),A4GNT (51146),AAAS (8086),AACS (65985),...,OR3A2 (4995),OR3A3 (8392),POLR2J (5439),PRAMEF10 (343071),PRR33 (102724536),RGPD2 (729857),SMIM10L3 (122526779),TPRX2 (503627),TTLL13 (440307),VCX2 (51480)
0,ACH-000029,0.000151,0.000541,-0.046740,0.040447,-0.188685,-0.035003,0.035252,-0.105373,0.150300,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACH-000030,0.169029,-0.182773,-0.023095,0.068845,-0.037532,-0.210192,0.131378,-0.396288,0.077450,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ACH-000074,-0.087069,-0.054984,0.211763,0.421891,0.013034,-0.065581,0.175972,0.240355,0.121743,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ACH-000090,0.205441,-0.138211,-0.140932,-0.139838,-0.036163,0.135382,0.052884,0.401293,0.283835,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ACH-000093,-0.133302,0.086477,-0.224933,0.024604,0.267048,-0.260316,0.223750,-0.025319,-0.015962,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### CRISPRGeneEffect — насколько клетка зависит от каждого гена

Эта таблица отвечает на вопрос: **что происходит с ростом и выживанием клеточной модели, когда экспериментально выключают определённый ген?** Выключение гена называют *нокаутом*. Это функциональный эксперимент, а не измерение количества РНК и не список естественных мутаций.

**Одна строка — одна клеточная модель. Один генный столбец — ген, который выключали. На пересечении — рассчитанная оценка Gene Effect.** В загруженном файле 18 531 генный столбец. Например, значение в строке `ACH-000552` и столбце `A1BG (1)` относится к выключению A1BG именно в этой модели.

| Столбец | Понятное описание | Использование |
|---|---|---|
| `Unnamed: 0` | Здесь это **не номер строки**, а идентификатор модели вида `ACH-000029`. Имя возникло из-за пустого заголовка в CSV. | Переименовать в `ModelID` и использовать как ключ объединения. В COAD-таблице это уже сделано. |
| Каждый столбец `SYMBOL (EntrezID)` | `SYMBOL` — название гена, число в скобках — его идентификатор NCBI Entrez. Например, `A1BG (1)` — один ген, а не два признака. Значение — безразмерный CRISPR Gene Effect, рассчитанный с помощью Chronos. | Может быть целевой переменной `y`, если предсказываем зависимость от выбранного гена. |

**Как понимать шкалу — условные примеры:**

| Значение | Интерпретация |
|---|---|
| Около `0` | Выключение гена оказывает небольшое влияние; ориентир — гены, не необходимые для выживания. |
| Около `−1` | Эффект сопоставим с типичным эффектом выключения общих необходимых для выживания генов. Это ориентир нормировки, а не «100% клеток погибло». |
| `−1.5` | Более сильный отрицательный эффект, чем `−0.3`; модель сильнее зависит от гена по этой оценке. |
| Положительное значение | Возможное преимущество роста после выключения либо экспериментальный/модельный шум. |
| `NaN` | Оценки нет. Это не нулевой эффект. |

Шкала не является вероятностью и не ограничена интервалом от −1 до 0. Нельзя автоматически объявить все значения ниже одного порога «летальными» без определения задачи.

**Для нашего датасета.** В COAD-подмножестве 48 моделей с CRISPR-профилем, тогда как в справочнике 74 модели. При левом объединении остальные модели сохраняются с пропусками в `crispr__`. Даже у модели с профилем отдельные гены могут иметь `NaN`. Если выбранный столбец становится `y`, строки с неизвестным `y` исключают из обучения, а само `y` не заполняют средним.

Экспрессия гена, мутация в нём и зависимость от его выключения — разные свойства. Высокая экспрессия сама по себе не доказывает сильную зависимость.

In [14]:
display("Model", Model.head())


'Model'

,ModelID,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,PatientSubtypeFeatures,...,PublicComments,CCLEName,HCMIID,PediatricModelType,ModelAvailableInDbgap,ModelSubtypeFeatures,WTSIMasterCellID,SangerModelID,COSMICID,ModelIDAlias
0,ACH-000001,PT-gj46wT,NIH:OVCAR-3,NIHOVCAR3,HGSOC,Ovary/Fallopian Tube,Ovarian Epithelial Tumor,High-Grade Serous Ovarian Cancer,HGSOC,NaN,...,NaN,NIHOVCAR3_OVARY,NaN,False,Approved for public sharing - CCLE,NaN,2201.0,SIDM00105,905933.0,NaN
1,ACH-000002,PT-5qa3uk,HL-60,HL60,AMLMRC,Myeloid,Acute Myeloid Leukemia,AML with Myelodysplasia-Related Changes,AMLMRC,"TP53(del), CDKN2A and NRAS mutations [PubMed=2...",...,NaN,HL60_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,"NRAS, BCOR and CDKN2A",55.0,SIDM00829,905938.0,NaN
2,ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,CACO2_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00891,NaN,NaN
3,ACH-000004,PT-q4K2cp,HEL,HEL,AMLNOS,Myeloid,Acute Myeloid Leukemia,"AML, NOS",AMLNOS,JAK2 and TP53 mutations,...,NaN,HEL_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,JAK2 and TP53,783.0,SIDM00594,907053.0,NaN
4,ACH-000005,PT-q4K2cp,HEL 92.1.7,HEL9217,AML,Myeloid,Acute Myeloid Leukemia,Acute Myeloid Leukemia,AML,JAK2 and TP53 mutations,...,NaN,HEL9217_HAEMATOPOIETIC_AND_LYMPHOID_TISSUE,NaN,True,Approved for public sharing - CCLE,NaN,NaN,SIDM00593,NaN,NaN


### Model — паспорт клеточных моделей

**Одна строка — одна модель, а не один пациент и не один эксперимент.** Здесь записаны название культуры, диагноз, сведения о доноре, происхождение и условия культивирования. Таблица нужна как справочник для остальных файлов.

Связь может выглядеть так: **один пациент → несколько моделей → несколько условий каждой модели → несколько результатов секвенирования**. Поэтому разные ModelID не всегда являются независимыми образцами от разных людей.

В этом ноутбуке отобраны 74 строки с `OncotreeCode == "COAD"`. Примеры ниже взяты из сохранённых COAD-таблиц. `NaN` означает отсутствие значения или неприменимость поля; текст `Unknown` / `unknown` тоже указывает на неизвестные сведения, но pandas не всегда автоматически превращает его в NaN.

| Столбец | Что означает | Как читать и использовать |
|---|---|---|
| `ModelID` | Уникальный ID модели DepMap (`ACH-xxxxxx`); основной ключ для объединения таблиц. | Пример: ACH-000003. Основной ключ к остальным таблицам; не числовой признак для ML. |
| `PatientID` | Идентификатор пациента/донора, с которым связана модель. | Пример: PT-puKIyc. Один донор может соответствовать нескольким моделям. CACO2 и C2BBe1 в ваших данных имеют один PatientID; это важно для группового train/test-разбиения. |
| `CellLineName` | Общепринятое название клеточной линии. | Человекочитаемое имя, например CACO2. Удобно для подписей; объединять надёжнее по ModelID. |
| `StrippedCellLineName` | Нормализованное имя без пробелов и специальных символов. | Имя без части разделителей и оформления. Это альтернативное написание названия, не новый биологический признак. |
| `DepmapModelType` | Сокращённый код типа/подтипа модели; обычно код OncoTree, иначе код DepMap. | В COAD-подмножестве значение COAD. Не путать с ModelType, где указаны Cell Line / Organoid. |
| `OncotreeLineage` | Верхнеуровневая линия/орган происхождения по OncoTree. | Широкая категория происхождения. Здесь Bowel — кишечник. |
| `OncotreePrimaryDisease` | Основное заболевание по OncoTree. | Следующий уровень классификации: Colorectal Adenocarcinoma — колоректальная аденокарцинома. |
| `OncotreeSubtype` | Детальный гистологический/молекулярный подтип заболевания. | Более конкретное название: Colon Adenocarcinoma — аденокарцинома ободочной кишки. |
| `OncotreeCode` | Код подтипа OncoTree; пусто, если подходящего кода нет. | Код диагноза. Именно равенство COAD использовано в этом ноутбуке для отбора моделей. |
| `PatientSubtypeFeatures` | Агрегированные известные особенности исходной опухоли пациента. | Текст о молекулярных особенностях опухоли донора. В сохранённом COAD-подмножестве поле пустое. |
| `RRID` | Идентификатор Cellosaurus/Research Resource Identifier. | Пример: CVCL_0025. Ссылка-идентификатор для поиска модели в Cellosaurus; не измерение. |
| `Age` | Возраст пациента/донора на момент взятия образца, в годах. | Возраст донора, например 72 года; не возраст культуры и не время эксперимента. NaN не равен 0 годам. |
| `AgeCategory` | Возрастная категория: Adult, Pediatric, Fetus или Unknown. | Категория возраста, например Adult. Unknown — неизвестно; это не ещё одна возрастная группа. |
| `Sex` | Зарегистрированный пол пациента/донора. | Категориальное поле Male / Female / Unknown. Unknown следует осмысленно обработать как неизвестное значение. |
| `PatientRace` | Указанная в клинических данных раса; не вычислена по геному. | Исходная категория из метаданных. Не заменяет измеренную генетическую ancestry; в файле встречаются разные регистры и unknown. |
| `PrimaryOrMetastasis` | Происхождение образца: первичная, метастатическая, рецидивная опухоль и т. п. | Primary — первичная опухоль; Metastatic — метастаз. Не описывает, способна ли культура метастазировать в эксперименте. |
| `SampleCollectionSite` | Анатомическое место, где был взят образец. | Откуда взяли образец. Например, liver у модели рака кишечника может означать метастаз в печени, а не первичный рак печени. |
| `SourceType` | Тип источника поступления модели: коммерческий, академический и др. | В этом файле встречаются ATCC, KCLB, DSMZ — источники/коллекции, откуда поступила модель. |
| `SourceDetail` | Дополнительные сведения об источнике модели. | Дополнительные сведения об источнике. Свободный текст; в COAD-подмножестве пусто. |
| `CatalogNumber` | Каталожный номер модели у поставщика. | Например HTB-37 — номер в каталоге поставщика. Хранить как строку. |
| `ModelType` | Тип модели при поступлении, например Cell Line или Organoid. | Cell Line — клеточная линия; Organoid — органоид. Это формат модели, а не диагноз. |
| `TissueOrigin` | Вид/происхождение ткани: Human, Mouse или Other. | Сведения о происхождении ткани. В ваших COAD-данных поле пустое, поэтому ничего по нему не выводим. |
| `ModelDerivationMaterial` | Материал, из которого получена модель: свежая ткань, PDX и др. | Материал, из которого получили модель. Отличается от органа взятия образца и поставщика. |
| `ModelTreatment` | Обработка модели, например вирусная трансформация. | Обработка при создании/подготовке модели; не следует автоматически считать это лечением пациента. |
| `PatientTreatmentStatus` | Отношение взятия образца к лечению пациента: до, во время или после лечения. | Сведения о лечении донора к моменту получения образца. Unknown означает отсутствие информации, а не отсутствие лечения. |
| `PatientTreatmentType` | Тип лечения пациента до/на момент взятия образца. | Тип предшествующего лечения пациента, если известен. Не текущая экспериментальная обработка культуры. |
| `PatientTreatmentDetails` | Подробности лечения пациента. | Подробности лечения текстом; требуют отдельной обработки, если понадобятся в X. |
| `Stage` | Стадия исходной опухоли/заболевания. | Стадия заболевания у пациента. Интерпретируется вместе со StagingSystem. |
| `StagingSystem` | Система стадирования, например AJCC Pathologic Stage. | Система, по которой записана стадия. Значения разных систем нельзя механически считать одной числовой шкалой. |
| `PatientTumorGrade` | Степень злокачественности или иной показатель пролиферации опухоли. | Grade характеризует свойства опухоли по принятой системе оценки; не то же самое, что стадия Stage. |
| `PatientTreatmentResponse` | Известный ответ пациента на лечение. | Описанный ответ пациента на лечение. Не измерение ответа этой культуры на CRISPR. |
| `GrowthPattern` | Формат роста при поступлении: Adherent, Suspension, Dome, Spheroid, Mixed и др. | Adherent — прикреплённый рост; Suspension — в суспензии; Mixed — смешанный. Это категориальный признак. |
| `OnboardedMedia` | Код среды, записанный в этой выгрузке. | В вашем CSV здесь коды вроде MF-015-009. Не воспринимать код как числовую величину. |
| `FormulationID` | Описание рецептуры среды, записанное в этой выгрузке. | Несмотря на название, здесь текст рецептуры: EMEM + 20% FBS, RPMI + 10% FBS. Смысл уточняем по содержимому файла. |
| `SerumFreeMedia` | Признак бессывороточной среды по классификации источника. | Флаг условий среды, не количество сыворотки. False не сообщает точную концентрацию; она может быть указана в рецептуре. |
| `PlateCoating` | Покрытие культуральной поверхности: laminin, Matrigel, collagen, none и т. п. | Покрытие поверхности для культивирования. Пропуск не доказывает отсутствие покрытия. |
| `EngineeredModel` | Признак искусственно модифицированной модели. | Флаг искусственной модификации модели. Пропуск нельзя автоматически превратить в False. |
| `EngineeredModelDetails` | Подробности генетической или иной модификации. | Описание модификации; уточняет EngineeredModel, если заполнено. |
| `CulturedResistanceDrug` | Препарат, к которому модель адаптировали для получения устойчивости. | Препарат, к устойчивости к которому адаптировали культуру. Не числовая оценка лекарственной чувствительности. |
| `PublicComments` | Публичные комментарии и важные примечания к модели. | Прочитать перед анализом: в вашем файле есть замечания о происхождении и родстве отдельных линий. |
| `CCLEName` | Стандартизованное имя линии в CCLE. | Например CACO2_LARGE_INTESTINE. Историческое имя CCLE; для join используйте ModelID. |
| `HCMIID` | Идентификатор Human Cancer Models Initiative. | Внешний идентификатор Human Cancer Models Initiative. Пустое поле не означает низкое качество модели. |
| `PediatricModelType` | Признак/категория педиатрической модели. | Флаг педиатрической категории модели. Не путать с форматом Cell Line / Organoid. |
| `ModelAvailableInDbgap` | Статус разрешения на публичный обмен геномными данными модели (dbGaP). | Метаданные доступности/категории обмена данными: встречаются Approved for public sharing - CCLE и HMB-MDS. Не биологический признак. |
| `ModelSubtypeFeatures` | Курируемые подтверждённые молекулярные особенности самой модели. | Особенности самой модели, например MSI — микросателлитная нестабильность. Пусто не означает MSS; признаки могут пересекаться с вашей будущей целью. |
| `WTSIMasterCellID` | Идентификатор модели Wellcome Trust Sanger Institute. | Внешний ID Sanger. Даже если pandas прочитал его как 569.0, это идентификатор, не измерение. |
| `SangerModelID` | Идентификатор модели в Sanger/Cell Model Passports. | Внешний ID вида SIDM00891; нужен для сопоставления с ресурсами Sanger. |
| `COSMICID` | Идентификатор клеточной линии/образца в COSMIC. | ID в COSMIC, например 907795.0 после чтения CSV. Не число мутаций. |
| `ModelIDAlias` | Прежние или альтернативные DepMap ModelID. | Прежние/альтернативные ModelID; иногда несколько через запятую. Не готовый одиночный ключ для merge. |

**Что брать в ML.** Не нужно автоматически помещать все 49 столбцов в X. Идентификаторы и названия обычно сохраняют для сопоставления и интерпретации. Категории требуют кодирования, текст — отдельной обработки. Диагностические поля, по которым уже отобран COAD, могут быть константными и не помогать различать модели. Сведения о молекулярном подтипе или ответе на лечение могут раскрывать будущую целевую переменную: их пригодность зависит от выбранного y.

Условия культуры и источник могут объяснять технические различия. Если цель — обобщение на новые независимые модели/доноров, полезно учитывать PatientID и известное родство линий при разбиении данных.

In [6]:
display("Omics Expression TPM Logp1 Human Protein Coding Genes", OmicsExpressionTPMLogp1HumanProteinCodingGenes.head())


'Omics Expression TPM Logp1 Human Protein Coding Genes'

,Unnamed: 0,SequencingID,ModelConditionID,ModelID,IsDefaultEntryForMC,IsDefaultEntryForModel,TSPAN6 (7105),TNMD (64102),DPM1 (8813),SCYL3 (57147),...,ATXN8 (724066),SMIM42 (117981789),NPBWR1 (2831),ACTL10 (170487),RNF228 (122319436),PANO1 (101927423),HRURF (120766137),PRRC2B (84726),F8A2 (474383),F8A1 (8263)
0,0,CDS-010xbm,MC-001113-k2lR,ACH-001113,Yes,Yes,4.956577,0.000000,7.577648,3.179411,...,0.0,0.0,0.414727,0.077634,1.113094,0.411901,0.0,5.134808,1.214541,4.315653
1,1,CDS-02TzJp,MC-001289-BpdI,ACH-001289,Yes,Yes,4.955015,0.617117,7.333933,2.782935,...,0.0,0.0,0.029809,0.000000,1.269267,1.026807,0.0,5.953793,0.006969,4.250636
2,2,CDS-0693hw,MC-001339-5nRN,ACH-001339,Yes,Yes,3.421952,0.000000,7.546069,2.615880,...,0.0,0.0,0.127768,0.105594,0.413829,0.540575,0.0,4.205971,0.030894,2.783448
3,3,CDS-07Plat,MC-001619-IR6I,ACH-001619,No,No,5.196729,0.000000,6.362268,2.144996,...,0.0,0.0,0.075396,0.932918,0.000000,0.511978,0.0,4.510747,3.223597,5.106596
4,4,CDS-08FOcu,MC-001979-E3qW,ACH-001979,Yes,Yes,4.651643,0.000000,5.946408,2.454515,...,0.0,0.0,0.012012,0.000000,1.006470,0.729221,0.0,4.922916,0.150926,4.661556


### OmicsExpressionTPMLogp1HumanProteinCodingGenes — активность генов на уровне РНК

Таблица отвечает на вопрос: **сколько РНК каждого белок-кодирующего гена обнаружено в образце?** Это оценка экспрессии по RNA-seq. Она не равна количеству белка и сама по себе не показывает, необходим ли ген для выживания клетки.

**Одна строка — профиль секвенирования.** Для одной модели в полном файле могут быть несколько профилей. После шести служебных столбцов идут 19 215 генных столбцов.

| Столбец | Что означает | Как использовать |
|---|---|---|
| `Unnamed: 0` | Сохранённый порядковый индекс строки CSV: 68, 115 и т. п. | Здесь это техническое число, в отличие от одноимённого столбца CRISPR. Не использовать для объединения. |
| `SequencingID` | ID конкретного результата секвенирования, например `CDS-2HrGbX`. | Различает профили. РНК и ДНК одной модели могут иметь разные SequencingID. |
| `ModelConditionID` | ID состояния/условий модели, например `MC-000999-JOmJ`. | Нужен для анализа на уровне условий. Сам ID не раскрывает состав среды или обработку. |
| `ModelID` | ID клеточной модели, например `ACH-000999`. | Основной ключ к Model и к итоговому датасету с одной строкой на модель. |
| `IsDefaultEntryForMC` | `Yes`: профиль выбран основным представителем конкретного ModelConditionID. `No`: не основной для этого условия. | Позволяет выбирать профили на уровне условий; не гарантирует одну строку на ModelID. |
| `IsDefaultEntryForModel` | `Yes`: профиль выбран основным представителем модели в целом. `No`: не основной для этой модели. | Для нашего объединения выбираем Yes и затем проверяем уникальность ModelID. No не означает плохое качество. |
| Все столбцы `SYMBOL (EntrezID)` | Например `TSPAN6 (7105)`: символ гена и его числовой ID. Значение — `log2(TPM + 1)`. | Это числовые признаки экспрессии; в общем датасете имеют префикс `expr__`. |

**Зачем нужны два флага default?** Условный пример, не реальные записи из файла:

| ModelID | Условие | Профиль | Default для MC | Default для модели |
|---|---|---|---|---|
| Модель A | Условие 1 | Профиль 1 | Yes | Yes |
| Модель A | Условие 1 | Профиль 2 | No | No |
| Модель A | Условие 2 | Профиль 3 | Yes | No |

Фильтр по MC оставит два профиля одной модели, фильтр по модели — один. Default — выбор представителя в выпуске данных, а не отметка «контрольная группа» или результат отдельной оценки качества нами.

**Как читать числа.** TPM (*Transcripts Per Million*) — относительная оценка количества РНК с нормировкой, учитывающей длину транскриптов и объём библиотеки. В файле применён логарифм `x = log2(TPM + 1)`, чтобы уменьшить разброс между очень низкими и очень высокими значениями.

| Значение в таблице x | Исходный TPM: `2**x - 1` |
|---|---|
| 0 | 0 |
| 1 | 1 |
| 3 | 7 |
| 5 | 31 |

`0` — нулевое оценённое TPM в этом измерении, а не отсутствие гена в ДНК. `NaN` — неизвестное значение. Разница в одну единицу соответствует удвоению **TPM + 1**, не строго TPM. Повторно логарифмировать эти значения не нужно.

**Для нашего датасета.** В сохранённой COAD-таблице 66 профилей, все отмечены default для модели. Остальные строки справочника после left join получат NaN в `expr__`. Встроенная нормировка TPM не отменяет возможных различий между партиями экспериментов. Импутацию, дополнительное масштабирование и отбор признаков для ML обучают только на train.

In [7]:
display("Omics Somatic Mutations", OmicsSomaticMutations.head())

'Omics Somatic Mutations'

,Unnamed: 0,SequencingID,ModelID,ModelConditionID,IsDefaultEntryForModel,IsDefaultEntryForMC,Chrom,Pos,Ref,Alt,...,GwasDisease,GwasPmID,GtexGene,ProveanPrediction,AMClass,AMPathogenicity,Rescue,RescueReason,Hotspot,EntrezGeneID
0,0,CDS-aN8PNg,ACH-000062,MC-000062-eiDD,Yes,Yes,chr1,818203,G,A,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,400728.0
1,1,CDS-qjpRtt,ACH-002059,MC-002059-DNM1,Yes,Yes,chr1,918916,CTGA,C,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,100130417.0
2,2,CDS-bOBkBi,ACH-000402,MC-000402-iC8K,Yes,Yes,chr1,924510,GC,AA,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0
3,3,CDS-52aoJ0,ACH-000693,MC-000693-vUEr,Yes,Yes,chr1,924657,C,G,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0
4,4,CDS-jBuyoZ,ACH-000930,MC-000930-v0gO,Yes,Yes,chr1,924750,C,T,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0


### OmicsSomaticMutations — какие изменения ДНК зарегистрированы в моделях

**Одна строка — один вариант (альтернативный аллель) в одном профиле секвенирования, а не одна клеточная модель.** В одной модели могут быть тысячи вариантов; в одном гене — несколько вариантов. Поэтому 161 838 строк COAD_Mutations не означают 161 838 независимых образцов для ML.

Таблица содержит обнаруженные и прошедшие отбор варианты, дополненные аннотациями. *Соматический* означает изменение, связанное с клетками ткани/опухоли, в отличие от наследуемого варианта. Сам факт присутствия в этой таблице не является доказательством, что изменение вызывает рак или делает клетку зависимой от гена.

**Три разных уровня сведений:**

- Наблюдение: где отличается ДНК и сколько чтений подтверждает отличие (`Chrom`, `Pos`, `Ref`, `Alt`, `AF`, счётчики).
- Аннотация: какой ген/транскрипт затронут и какое изменение предсказывается (`HugoSymbol`, `ProteinChange`, `VepImpact`).
- Внешние знания и прогнозы: что сообщают базы и алгоритмы (`CivicDescription`, `LikelyLoF`, `AMClass` и другие).

Примеры формата ниже взяты из ваших CSV; биологические трактовки поясняют тип поля, а не устанавливают эффект конкретного варианта.

#### 1. К какой модели и профилю относится вариант

| Столбец | Что означает | Как читать и использовать |
|---|---|---|
| `Unnamed: 0` | Технический индекс строки CSV; биологического смысла не имеет. | Технический номер строки. Не ключ варианта и не ML-признак. |
| `SequencingID` | Уникальный ID результата секвенирования (`CDS-...`). | Один CDS-ID повторяется для множества вариантов одного профиля. Не объединять с RNA-seq по этому ID без проверки соответствия. |
| `ModelID` | ID базовой модели DepMap (`ACH-...`), ключ к `Model.csv`. | ACH-ID связывает вариант с моделью. Здесь он многократно повторяется — прямой merge размножит строки. |
| `ModelConditionID` | ID условия/состояния модели во время анализа (`MC-...`). | MC-ID различает условия модели. Условия измерения ДНК и РНК могут различаться даже при одинаковом ModelID. |
| `IsDefaultEntryForModel` | Является ли профиль представителем базовой модели. | Yes выбирает основной профиль модели, но оставляет все варианты этого профиля: это по-прежнему много строк на модель. |
| `IsDefaultEntryForMC` | Является ли профиль представителем данного состояния модели. | Yes выбирает основной профиль конкретного условия. Разницу двух флагов показывает пример под таблицей экспрессии. |

#### 2. Где вариант найден и какие чтения его поддерживают

| Столбец | Что означает | Как читать и использовать |
|---|---|---|
| `Chrom` | Хромосома варианта в используемой сборке референса. | Например chr1. Категория местоположения; координаты сопоставимы только в одной сборке генома. |
| `Pos` | Геномная координата начала варианта (1-based). | Например 942225. Не уникальна без Chrom, Ref и Alt. Перед сопоставлением с BED/VCF проверить сборку и систему координат. |
| `Ref` | Референсный аллель. | Последовательность референса в данном месте; не обязательно аллель, преобладающий в клеточной модели. |
| `Alt` | Альтернативный (наблюдаемый) аллель. | Последовательность варианта. Ref=G и Alt=A — замена; более длинные строки могут описывать вставку, удаление или сложную замену. |
| `AF` | Оценённая доля альтернативного аллеля в секвенированном образце. | Около 0.4 означает оценённую долю альтернативного аллеля около 40%. Это не 40% пациентов и не вероятность патогенности. Из-за способа расчёта может отличаться от AltCount/DP. |
| `DP` | Приблизительная глубина чтений в позиции с учётом особенностей обработки. | Глубина: сколько чтений учитывается в позиции. При малой глубине оценка AF менее устойчива; фильтрация чтений может влиять на соотношение счётчиков. |
| `RefCount` | Число чтений, поддерживающих Ref. | Например 17 чтений поддерживают референс. Чтение — фрагмент результата секвенирования, не отдельная клетка. |
| `AltCount` | Число чтений, поддерживающих Alt. | Например 26 чтений поддерживают вариант. Это подтверждение наблюдения, а не число разных мутаций. |
| `GT` | Генотип в нотации VCF, например `0/1`. | 0 обозначает Ref, 1 — первый Alt. 0/1 означает наличие обоих; вертикальный разделитель в 0 / 1 обозначает фазирование. У опухолевых моделей GT не заменяет анализ числа копий. |
| `PS` | Phase Set: ID блока фазирования генотипа, если доступен. | Группа вариантов с совместным фазированием: информация об их расположении на копиях хромосомы. Значение не является score. |
| `VariantType` | Физический тип варианта: SNV, insertion, deletion и т. п. | SNV — замена одной буквы ДНК; insertion/deletion — вставка/удаление; substitution может описывать замену нескольких букв. |

#### 3. Что изменилось в гене, транскрипте и белке

| Столбец | Что означает | Как читать и использовать |
|---|---|---|
| `VariantInfo` | Наиболее релевантные последствия VEP, возможно несколько через `&`. | missense_variant — замена аминокислоты; frameshift_variant — сдвиг рамки считывания; splice_acceptor_variant — изменение участка сплайсинга. Несколько последствий могут соединяться через &. |
| `DNAChange` | Изменение на уровне транскрипта в HGVS-нотации (`c.` или `n.`). | Например ENST…:c.251G>A: в нумерации кодирующей последовательности транскрипта позиция 251 меняется с G на A. Это не координата Pos. |
| `ProteinChange` | Изменение белка в HGVS-нотации (`p.`), если применимо. | p.R84H: аргинин R в позиции 84 заменён на гистидин H. В p.M53WfsTer8 fs означает сдвиг рамки, Ter — стоп-сигнал. Пропуск не доказывает отсутствие эффекта. |
| `HugoSymbol` | Официальный символ затронутого гена. | Например SAMD11. Именно по этому полю в последней ячейке формируются бинарные признаки mut__. Названия генов могут иметь алиасы. |
| `Exon` | Номер затронутого экзона/общее число экзонов. | 9/14 читается как девятый экзон из четырнадцати у аннотированного транскрипта. Это строка, не дробь. |
| `Intron` | Номер затронутого интрона/общее число интронов. | 4/17 — четвёртый интрон из семнадцати. Экзоны входят в зрелую РНК, интроны обычно удаляются при сплайсинге. |
| `EnsemblGeneID` | Идентификатор гена Ensembl (`ENSG...`). | ENSG… — идентификатор гена; полезен для сопоставления аннотаций. |
| `EnsemblFeatureID` | ID выбранного VEP транскрипта/feature, обычно `ENST...`. | Обычно ENST… — идентификатор транскрипта. Один ген может иметь несколько транскриптов и разные последствия одного варианта. |
| `HgncName` | Полное утверждённое HGNC название гена. | Полное название удобно для чтения. Для объединения лучше использовать согласованные идентификаторы. |
| `HgncFamily` | Семейство/семейства гена по HGNC. | Может содержать несколько групп через разделитель. Это не одна числовая характеристика. |
| `UniprotID` | Идентификатор белка UniProt. | ID белка, иногда с указанием изоформы, например Q494U1-1. Не идентификатор модели. |
| `EntrezGeneID` | Числовой идентификатор затронутого гена в NCBI Entrez Gene. | Например 148398.0 после чтения CSV. Тот же тип ID, что указан в скобках генных столбцов экспрессии/CRISPR; полезен для согласования генов. |

#### 4. Дополнительные аннотации последовательности и идентификаторы

| Столбец | Что означает | Как читать и использовать |
|---|---|---|
| `DbsnpRsID` | Идентификатор rs в dbSNP, иногда сохранённый только числом. | В вашем CSV встречается 1488554459.0 — номер rs без префикса, прочитанный как float. Наличие в dbSNP не означает безвредность. |
| `GcContent` | Доля GC в локальном контексте варианта. | 0.75 означает примерно 75% G и C в рассматриваемом контексте. Не доля мутантного аллеля. |
| `NMD` | Аннотация о предполагаемом nonsense-mediated decay для затронутых транскриптов. | Nonsense-mediated decay — механизм удаления некоторых РНК с преждевременным стоп-сигналом. Здесь структурированная аннотация о генах/транскриптах, а не измерение количества РНК. |
| `MolecularConsequence` | Список молекулярных последствий с кодами Sequence Ontology. | Например SO:0001583 вместе с missense_variant. Это код и название последствия; встречаются списки, а не одно значение. |
| `VepImpact` | Категория тяжести VEP: HIGH, MODERATE, LOW или MODIFIER. | HIGH / MODERATE / LOW / MODIFIER — категории предполагаемого эффекта на последовательность. HIGH не равен доказанному cancer driver; MODIFIER не означает доказанную безвредность. |
| `VepBiotype` | Биотип выбранного транскрипта, например protein_coding или lncRNA. | protein_coding — белок-кодирующий транскрипт; lncRNA — длинная некодирующая РНК. Помогает определить, осмыслен ли анализ белкового эффекта. |
| `VepHgncID` | Идентификатор HGNC из VEP (`HGNC:...`). | Например HGNC:28706. Ещё один идентификатор того же гена в другой системе. |
| `VepExistingVariation` | Совпавшие известные варианты из источников VEP. | Список совпавших известных вариантов, например rs… и COSV…. Это ссылки в базы, не отдельные наблюдения этой модели. |
| `VepManeSelect` | MANE Select транскрипт и версия, если выбранная аннотация ему соответствует. | Ссылка на согласованный представительский транскрипт MANE; в файле значения вроде NM_001385641.1. Суффикс после точки — версия. |
| `VepENSP` | Идентификатор белка Ensembl (`ENSP...`). | ENSP… — белок в Ensembl. Не путать ENSG (ген), ENST (транскрипт) и ENSP (белок). |
| `VepSwissprot` | Идентификатор рецензируемой записи Swiss-Prot. | Ссылка на аннотированный белок Swiss-Prot. Поле может содержать версионную часть. |

#### 5. Прогнозы эффекта и популяционные частоты

| Столбец | Что означает | Как читать и использовать |
|---|---|---|
| `Sift` | Предсказание SIFT для missense-варианта, часто класс и score. | Пример tolerated(0.06): текстовый класс и число в одной строке. Перед числовым анализом нужно распарсить; меньшее значение обычно указывает на более неблагоприятный прогноз. |
| `Polyphen` | Предсказание PolyPhen для missense-варианта, часто класс и score. | Например possibly_damaging(0.783). Здесь направление шкалы другое: большее значение обычно соответствует более неблагоприятному прогнозу. |
| `GnomadeAF` | Частота аллеля в экзомах gnomAD. | Частота варианта среди аллелей в популяционной базе экзомов, а не в вашей модели. Не подменяет AF. Ноль/пропуск не доказывает, что вариант существует только в опухоли. |
| `GnomadgAF` | Частота аллеля в геномах gnomAD. | Аналогичная популяционная частота по полным геномам. Отличается от GnomadeAF составом и типом исходных данных. |
| `VepClinSig` | Клиническая значимость из источников VEP/ClinVar. | Например uncertain_significance — значение не определено; pathogenic — указан патогенный статус. Интерпретация относится к контексту базы и не равна CRISPR-зависимости. |
| `VepSomatic` | Признак, что вариант отмечен как соматический во внешних источниках VEP. | Может быть 0&1: статусы нескольких совпавших внешних записей. Не преобразовывать всю строку в один bool и не использовать как единственное доказательство соматического происхождения. |
| `VepPliGeneValue` | pLI гена: оценка непереносимости loss-of-function вариантов. | Генная характеристика непереносимости потери функции. Не вероятность того, что конкретный вариант вызывает рак. |
| `VepLofTool` | LoFtool score — предсказанная чувствительность гена к loss of function. | Оценка гена, связанная с непереносимостью потери функции; не измерение эффекта варианта в этой клеточной линии. |
| `OncogeneHighImpact` | Флаг high-impact варианта в известном онкогене. | Сочетает принадлежность гена к онкогенам и высокий аннотированный эффект варианта. Не доказывает активацию этого онкогена. |
| `TumorSuppressorHighImpact` | Флаг high-impact варианта в гене-супрессоре опухоли. | Сочетает принадлежность к генам-супрессорам опухоли и высокий эффект варианта. Не доказывает выключение всех копий гена. |
| `TranscriptLikelyLof` | Транскрипт(ы), где вариант прогнозируется как вероятный LoF. | Список ENST-ID, иногда через точку с запятой. Не логический флаг, в отличие от LikelyLoF. |
| `Brca1FuncScore` | Экспериментальная функциональная оценка BRCA1 для соответствующих вариантов. | Специализированная экспериментальная оценка для вариантов BRCA1. Не универсальная шкала для всех генов; многие пропуски ожидаемы. |
| `LikelyLoF` | Итоговый флаг вероятной потери функции по правилам пайплайна DepMap. | LoF = loss of function, потеря функции. True — прогноз/аннотация вероятной потери функции по правилам пайплайна, не экспериментальное доказательство. |
| `RevelScore` | Прогноз эффекта missense-варианта алгоритмом REVEL. | Оценка эффекта missense-варианта. Более высокое значение указывает на более неблагоприятный прогноз; не считать калиброванной вероятностью зависимости клетки от гена. |
| `ProveanPrediction` | Предсказание PROVEAN о влиянии варианта на функцию белка. | Neutral / Damaging — классы прогноза влияния на белок. Прогноз может расходиться с другими алгоритмами. |
| `AMClass` | Класс AlphaMissense: likely pathogenic, likely benign или ambiguous. | likely_benign — вероятно безвредный; likely_pathogenic — вероятно патогенный; ambiguous — неопределённый прогноз AlphaMissense. |
| `AMPathogenicity` | Непрерывная оценка патогенности AlphaMissense (выше — патогеннее). | Числовой score AlphaMissense. Применим к соответствующим missense-вариантам; NaN не равно нулевой патогенности. |

#### 6. Внешние знания о варианте и правила его сохранения

| Столбец | Что означает | Как читать и использовать |
|---|---|---|
| `CivicID` | Идентификатор клинической интерпретации варианта в CIViC. | ID записи в базе CIViC. Не оценка чувствительности к лекарству. |
| `CivicDescription` | Текстовое описание клинического свидетельства CIViC. | Текст внешних свидетельств. Он может относиться к другому заболеванию или эксперименту, а не к текущей модели. |
| `CivicScore` | Агрегированная оценка уровня/объёма свидетельств CIViC. | Оценка записи/свидетельств CIViC. Не вероятность патогенности и не измеренный ответ модели. |
| `HessDriver` | Предсказанный driver-статус по методу Hess и соавторов. | Driver — вариант, потенциально способствующий развитию опухоли. В вашем файле встречаются True, False и Y: перед ML нужно явно согласовать кодирование. |
| `HessSignature` | Мутационная сигнатура/категория, связанная с Hess driver-аннотацией. | Пример UV:1,POLE:5. Внешняя аннотация по Hess, а не рассчитанные доли мутационных сигнатур конкретной модели. |
| `PharmgkbId` | Идентификатор фармакогеномной аннотации PharmGKB. | Ссылка на фармакогеномную базу. Наличие ID само по себе не сообщает, чувствительна ли модель к препарату. |
| `GwasDisease` | Заболевание/признак из совпавшей GWAS-аннотации. | Признак из внешнего GWAS, например Body mass index. Это не диагноз пациента, от которого получена линия. |
| `GwasPmID` | PubMed ID публикации GWAS. | PubMed-ID публикации, например 30595370.0 после чтения CSV. Это ссылка, не измерение. |
| `GtexGene` | Связанный ген из GTEx/eQTL-аннотации. | Ген, экспрессия которого связана с вариантом в данных GTEx. Это не значение экспрессии из нашей RNA-seq-таблицы. |
| `Rescue` | Флаг варианта, который пайплайн «спас» от стандартной фильтрации. | True означает сохранение варианта по специальным критериям пайплайна. Это не эксперимент спасения клеток после нокаута. |
| `RescueReason` | Причина сохранения (`Rescue`) варианта. | Например OncoKB или TS_high_impact. Может быть несколько причин; поясняет Rescue. |
| `Hotspot` | Флаг известной рекуррентной hotspot-мутации. | Аннотированный hotspot по критериям выпуска. Не означает, что этот вариант частый именно среди наших 74 COAD-моделей. |

#### Пример чтения строки

В первой показанной COAD-записи: `ModelID = ACH-000969`, `Chrom = chr1`, `Pos = 942225`, `Ref = TG`, `Alt = GC`, `AF ≈ 0.595`, `DP = 43`, `RefCount = 17`, `AltCount = 26`. Это зарегистрированная замена последовательности в профиле данной модели. Значение `26/43 ≈ 0.605` близко к AF, но не равно ему: AF следует читать как отдельную оценку из исходного пайплайна, а не пересчитывать без необходимости.

Для объяснения белкового эффекта отдельно смотрят `ProteinChange` и аннотации. Для вопроса «нужен ли ген этой клетке для выживания?» нужна таблица CRISPR, а не только эта строка.

#### Пропуски, флаги и повторные записи

- `NaN` в прогнозе может означать, что алгоритм неприменим к типу варианта, нет аннотации или данных. Это не отрицательный прогноз.
- Строки `"False"`, `"No"`, `"Y"` нельзя бездумно преобразовывать через `astype(bool)`: непустая строка в Python считается True. Нужна явная таблица соответствий, сохраняющая неизвестные значения.
- `IsDefaultEntryForModel == "Yes"` выбирает профиль, но не одну мутацию. Для одной строки на модель варианты всё равно нужно агрегировать.
- Для проверки повторов используют профиль и координаты варианта, например `SequencingID + Chrom + Pos + Ref + Alt`; одного `ModelID` или `HugoSymbol` недостаточно.

#### Что делает последняя ячейка ноутбука

Она оставляет default-записи модели и превращает пары `ModelID + HugoSymbol` в широкую таблицу: один столбец `mut__GENE` на ген.

| Итоговое значение | Что оно означает |
|---|---|
| 1 | Есть хотя бы одна запись варианта этого гена в выбранном профиле модели. Два варианта в одном гене всё равно дают 1. |
| 0 | Для модели есть записи мутаций, но для этого гена записи нет. Это не гарантия достаточного покрытия гена и не подтверждение wild-type. |
| NaN | У модели нет выбранных записей мутаций; по этому файлу нельзя отличить отсутствие измерения от отсутствия обнаруженных вариантов. |

Это начальное представление «любой зарегистрированный вариант». Оно **не различает** missense, потерю функции и hotspot и не сохраняет AF каждого варианта. Для другой ML-задачи можно строить отдельные признаки по типам последствий, LikelyLoF или Hotspot, но их смысл будет другим. Число строк нельзя автоматически называть TMB: для мутационной нагрузки на мегабазу нужны определение учитываемых вариантов и размер исследованной территории генома.

При объединении по ModelID мы собираем информацию на уровне модели. Это не гарантирует, что ДНК, РНК и CRISPR измеряли одновременно и в одинаковых условиях.

#### Где уточнять определения

Редкие поля и правила отбора сверены с [документацией мутационного пайплайна DepMap 26Q1](https://storage.googleapis.com/shared-portal-files/Tools/26Q1_Mutation_Pipeline_Documentation.pdf): в частности, AF — оценка доли аллеля, DP — приблизительная глубина; NMD и MolecularConsequence содержат аннотации, а не результаты функционального эксперимента. Правила и базы обновляются между выпусками. Версия локальных CSV в ноутбуке не указана, поэтому пороги и версии программ из 26Q1 здесь не приписываются вашим файлам. Для воспроизводимости следует сохранить название исходного выпуска и его документацию.

# Select COAD

In [8]:
# 1. Модели Colon Adenocarcinoma (COAD)
COAD_Model = Model.loc[
    Model["OncotreeCode"].eq("COAD")
].copy()

# Список идентификаторов моделей
coad_model_ids = set(COAD_Model["ModelID"])

print(f"Количество моделей COAD: {len(COAD_Model)}")
display(COAD_Model.head())

Количество моделей COAD: 74


,ModelID,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,PatientSubtypeFeatures,...,PublicComments,CCLEName,HCMIID,PediatricModelType,ModelAvailableInDbgap,ModelSubtypeFeatures,WTSIMasterCellID,SangerModelID,COSMICID,ModelIDAlias
2,ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,CACO2_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00891,NaN,NaN
6,ACH-000007,PT-NOXwpH,LS513,LS513,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,LS513_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,569.0,SIDM00677,907795.0,ACH-001078
8,ACH-000009,PT-puKIyc,C2BBe1,C2BBE1,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,C2BBE1_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,2104.0,SIDM01233,910700.0,NaN
87,ACH-000089,PT-0joF2E,NCI-H684,NCIH684,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,NCIH684_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM01665,NaN,NaN
200,ACH-000202,PT-UB7otW,COLO-320,COLO320,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,COLO320_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00071,NaN,NaN


In [9]:
# Filter tables

# 2. CRISPR Gene Effect
COAD_CRISPRGeneEffect = CRISPRGeneEffect.loc[
    CRISPRGeneEffect["Unnamed: 0"].isin(coad_model_ids)
].copy()

# Переименуем безымянный столбец для удобства
COAD_CRISPRGeneEffect = COAD_CRISPRGeneEffect.rename(
    columns={"Unnamed: 0": "ModelID"}
)

# 3. Экспрессия генов
COAD_Expression = OmicsExpressionTPMLogp1HumanProteinCodingGenes.loc[
    OmicsExpressionTPMLogp1HumanProteinCodingGenes["ModelID"].isin(coad_model_ids)
].copy()

# 4. Соматические мутации
COAD_Mutations = OmicsSomaticMutations.loc[
    OmicsSomaticMutations["ModelID"].isin(coad_model_ids)
].copy()

In [12]:
print("Model:", COAD_Model.shape)
display(COAD_Model.head())

print("CRISPR:", COAD_CRISPRGeneEffect.shape)
display(COAD_CRISPRGeneEffect.head())

print("Expression:", COAD_Expression.shape)
display(COAD_Expression.head())

print("Mutations:", COAD_Mutations.shape)
display(COAD_Mutations.head())

Model: (74, 49)


,ModelID,PatientID,CellLineName,StrippedCellLineName,DepmapModelType,OncotreeLineage,OncotreePrimaryDisease,OncotreeSubtype,OncotreeCode,PatientSubtypeFeatures,...,PublicComments,CCLEName,HCMIID,PediatricModelType,ModelAvailableInDbgap,ModelSubtypeFeatures,WTSIMasterCellID,SangerModelID,COSMICID,ModelIDAlias
2,ACH-000003,PT-puKIyc,CACO2,CACO2,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,CACO2_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00891,NaN,NaN
6,ACH-000007,PT-NOXwpH,LS513,LS513,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,LS513_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,569.0,SIDM00677,907795.0,ACH-001078
8,ACH-000009,PT-puKIyc,C2BBe1,C2BBE1,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,C2BBE1_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,2104.0,SIDM01233,910700.0,NaN
87,ACH-000089,PT-0joF2E,NCI-H684,NCIH684,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,NCIH684_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM01665,NaN,NaN
200,ACH-000202,PT-UB7otW,COLO-320,COLO320,COAD,Bowel,Colorectal Adenocarcinoma,Colon Adenocarcinoma,COAD,NaN,...,NaN,COLO320_LARGE_INTESTINE,NaN,False,Approved for public sharing - CCLE,NaN,NaN,SIDM00071,NaN,NaN


CRISPR: (48, 18532)


,ModelID,A1BG (1),A1CF (29974),A2M (2),A2ML1 (144568),A3GALT2 (127550),A4GALT (53947),A4GNT (51146),AAAS (8086),AACS (65985),...,OR3A2 (4995),OR3A3 (8392),POLR2J (5439),PRAMEF10 (343071),PRR33 (102724536),RGPD2 (729857),SMIM10L3 (122526779),TPRX2 (503627),TTLL13 (440307),VCX2 (51480)
27,ACH-000552,0.100104,0.091508,-0.084647,0.127559,0.104604,0.037875,0.283021,-0.240823,-0.040064,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52,ACH-000958,-0.165650,0.029107,-0.058589,0.121673,0.025537,-0.276521,0.020071,-0.093698,-0.191400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55,ACH-001061,-0.137968,0.000610,0.036002,0.029637,-0.121996,0.142163,0.040816,-0.701287,-0.089461,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
90,ACH-002669,-0.009169,-0.063475,-0.034713,0.148013,0.001379,0.023288,0.157885,-0.104796,0.004289,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
116,ACH-000403,0.021107,0.012370,-0.025229,-0.061882,0.001753,0.047060,0.142353,-0.394299,-0.065290,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Expression: (66, 19221)


,Unnamed: 0,SequencingID,ModelConditionID,ModelID,IsDefaultEntryForMC,IsDefaultEntryForModel,TSPAN6 (7105),TNMD (64102),DPM1 (8813),SCYL3 (57147),...,ATXN8 (724066),SMIM42 (117981789),NPBWR1 (2831),ACTL10 (170487),RNF228 (122319436),PANO1 (101927423),HRURF (120766137),PRRC2B (84726),F8A2 (474383),F8A1 (8263)
68,68,CDS-2HrGbX,MC-000999-JOmJ,ACH-000999,Yes,Yes,4.762301,0.486458,6.793519,2.585237,...,0.0,0.0,0.010904,2.002879,0.000000,0.821495,0.0,6.354614,1.044310,4.363143
115,115,CDS-46Q1Af,MC-000552-bdsQ,ACH-000552,Yes,Yes,4.092850,0.000000,7.012323,3.158833,...,0.0,0.0,0.000000,1.766875,0.015205,0.514804,0.0,6.048471,0.000000,5.173108
121,121,CDS-4NFXqK,MC-000501-66hN,ACH-000501,Yes,Yes,4.860501,0.030150,6.565561,2.237551,...,0.0,0.0,0.049699,1.270660,0.013951,1.037916,0.0,5.781955,2.692112,4.208322
129,129,CDS-4cuM53,MC-000798-O7D5,ACH-000798,Yes,Yes,4.990369,0.065611,5.550125,2.192759,...,0.0,0.0,0.013485,2.155127,0.021615,0.284616,0.0,5.126436,0.000000,4.924779
159,159,CDS-5dthvI,MC-000998-U8AM,ACH-000998,Yes,Yes,4.848172,0.338440,5.715350,2.312041,...,0.0,0.0,0.010203,2.235853,0.923017,1.335380,0.0,5.393403,0.000000,5.109202


Mutations: (161838, 69)


,Unnamed: 0,SequencingID,ModelID,ModelConditionID,IsDefaultEntryForModel,IsDefaultEntryForMC,Chrom,Pos,Ref,Alt,...,GwasDisease,GwasPmID,GtexGene,ProveanPrediction,AMClass,AMPathogenicity,Rescue,RescueReason,Hotspot,EntrezGeneID
20,20,CDS-vHArJs,ACH-000969,MC-000969-drm8,Yes,Yes,chr1,942225,TG,GC,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,148398.0
68,68,CDS-kmfiCf,ACH-000957,MC-000957-Yckn,Yes,Yes,chr1,961336,GC,G,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,339451.0
72,72,CDS-vHArJs,ACH-000969,MC-000969-drm8,Yes,Yes,chr1,961436,G,A,...,NaN,NaN,NaN,Neutral,likely_benign,0.1782,False,NaN,False,339451.0
85,85,CDS-mMTpJn,ACH-000963,MC-000963-1AeG,Yes,Yes,chr1,962894,C,G,...,NaN,NaN,NaN,Neutral,likely_benign,0.1319,False,NaN,False,339451.0
94,94,CDS-vHArJs,ACH-000969,MC-000969-drm8,Yes,Yes,chr1,965007,TG,T,...,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,339451.0


In [ ]:
# COAD_Model.to_csv("COAD_Model.csv", index=False)
# COAD_CRISPRGeneEffect.to_csv("COAD_CRISPRGeneEffect.csv", index=False)
# COAD_Expression.to_csv("COAD_Expression.csv", index=False)
# COAD_Mutations.to_csv("COAD_Mutations.csv", index=False)